# 03 - Assign cluster labels to ALL particles  (Stage 3)

Loads the model saved by notebook 1.5 (`data/kmeans_models/manual_1.5_k<K>/`)
with its own scaler, and labels every `complete` and `partial` particle by
nearest centroid. `no_gate` / `no_cross` get `cluster_label = -1`. With
`config.GROUP_MAP` set, `cluster_group` holds the merged groups.

**Output:** `data/labeled_trajectories.parquet` - one row per particle, with
`store` / `row` / `traj` to find the real particle in its Zarr store.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print(f"crossing: {C.LAT_CROSS} N | radii (km): {C.RADII_KM}")

In [ ]:
BEST_K = 70          # the K saved by notebook 1.5 (papermill: -p BEST_K <k>)

In [ ]:
import joblib
MODEL_DIR = C.manual_model_dir(BEST_K)
km     = joblib.load(MODEL_DIR / f"kmeans_k{BEST_K}.pkl")
scaler = joblib.load(MODEL_DIR / "scaler.pkl")
print("loaded", MODEL_DIR.name, "with", km.n_clusters, "clusters")

W = P.feature_weight_vector()
assert np.allclose(np.load(MODEL_DIR / "weights_W.npy"), W), (
    "config.RADIUS_WEIGHTS does not match the weights this model was fit with "
    "(paste the RADIUS_WEIGHTS printed by notebook 1.5 into config.py)")
print("config.RADIUS_WEIGHTS matches the fit:", C.RADIUS_WEIGHTS)

## 3.1  Predict

In [ ]:
features = pd.read_parquet(C.FEATURES_FILE)
labelable = features.status.isin(["complete", "partial"]).to_numpy()
X_all = scaler.transform(P.build_feature_matrix(features[labelable])) * W
labels = np.full(len(features), -1, dtype=np.int32)
labels[labelable] = km.predict(X_all).astype(np.int32)
features["cluster_label"] = labels
print(features.cluster_label.value_counts().sort_index().to_string())

## 3.2  Merge into groups (optional)

In [ ]:
if C.GROUP_MAP:
    groups, raw2grp = P.apply_group_map(features.cluster_label.to_numpy(), C.GROUP_MAP, BEST_K)
    features["cluster_group"] = groups.astype(np.int32)
    print("raw cluster -> group:", raw2grp)
    print(features.loc[features.cluster_group >= 0, "cluster_group"]
          .map(lambda g: C.GROUP_NAMES.get(g, g)).value_counts().to_string())
else:
    features["cluster_group"] = features["cluster_label"]
    print("GROUP_MAP empty -> cluster_group mirrors cluster_label")

## 3.3  Save

In [ ]:
features.to_parquet(C.LABELED_FILE, index=False)
lbl = features.cluster_label >= 0
print(f"labelled {lbl.sum():,} particles into {BEST_K} clusters; "
      f"{(~lbl).sum():,} unlabelled ({features.loc[~lbl, 'status'].value_counts().to_dict()})")
print("saved", features.shape, "->", C.LABELED_FILE)